# OrganAMNIST classification with AlexNet

Same data and same training procedure as the MLP notebook, so the comparison is fair.
Only the model and the input resolution change.

- **Input:** resized to 224x224 (AlexNet's stride-4 11x11 first conv needs the room).
- **Model:** AlexNet from scratch, ~34M parameters.
- **Training:** Adam, cross-entropy, early stopping patience 5, LR scheduler on plateau.

> **Runtime.** Measured at **3.5 min per training epoch** on an M1 GPU (0.77 s per
> batch of 128), plus ~40 s for validation. With early stopping this usually lands
> around 10-15 epochs, so budget **45-70 min** locally. That is workable, but the
> machine will get hot. Run the final version on UCloud or Colab; set `N_EPOCHS = 1`
> here just to check the pipeline.

In [ ]:
import os
# moved into notebooks/; walk up to the repo root so src/ and data/ resolve regardless of launch cwd
while not os.path.exists("requirements.txt") and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir("..")


In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from src.data import load_medmnist
from src.metrics import multiclass_metrics, plot_confusion, plot_history
from src.models import MLP, AlexNet
from src.seed import get_device, set_seed
from src.training import evaluate, fit

set_seed(27)
device = get_device()
print("device:", device)

In [ ]:
BATCH_SIZE = 128
LR = 5e-4
N_EPOCHS = 30
PATIENCE = 5
IMAGE_SIZE = 224

train_loader, val_loader, test_loader, info = load_medmnist(
    "organamnist", batch_size=BATCH_SIZE, image_size=IMAGE_SIZE
)
n_classes = len(info["label"])
n_channels = info["n_channels"]

print(f"task: {info['task']}  classes: {n_classes}  channels: {n_channels}")
print(f"train {len(train_loader.dataset)} | val {len(val_loader.dataset)} | test {len(test_loader.dataset)}")

In [ ]:
import random

images, labels = next(iter(train_loader))
class_names = [info["label"][str(i)] for i in range(n_classes)]

fig, axes = plt.subplots(4, 4, figsize=(7, 7))
for ax, idx in zip(axes.ravel(), random.sample(range(len(images)), 16)):
    ax.imshow(images[idx].squeeze() * 0.5 + 0.5, cmap="gray")
    ax.set_title(class_names[labels[idx].item()], fontsize=7)
    ax.axis("off")
fig.tight_layout()

In [ ]:
model = AlexNet(in_channels=n_channels, num_classes=n_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

print(f"trainable parameters: {sum(p.numel() for p in model.parameters()):,}")

## Train

The scheduler halves the learning rate when validation loss plateaus for 2 epochs;
early stopping then cuts training off at 5. Patience 2 < 5 on purpose, so the model
gets a chance to recover on a smaller LR before we give up on it.

In [ ]:
history = fit(model, train_loader, val_loader, criterion, optimizer, device,
              n_epochs=N_EPOCHS, patience=PATIENCE, scheduler=scheduler)

plot_history(history)

## Test

In [ ]:
test_acc, test_loss, y_true, y_prob = evaluate(model, test_loader, criterion, device)

print(f"Test accuracy: {test_acc:.2f}%   Test loss: {test_loss:.4f}")
for name, value in multiclass_metrics(y_true, y_prob).items():
    print(f"{name:>10}: {value:.4f}")

fig, ax = plt.subplots(figsize=(8, 8))
plot_confusion(y_true, y_prob.argmax(1), class_names=class_names, ax=ax, normalize="true")
plt.xticks(rotation=45, ha="right")
fig.tight_layout()

## Compare against the MLP

Fill this in once both notebooks have run:

| Model | Parameters | Test accuracy | Macro F1 | Macro AUC |
|---|---|---|---|---|
| MLP (784-512-11) | | | | |
| AlexNet | | | | |

Questions worth answering in your report:

- AlexNet has ~40x more parameters than the MLP. Is the accuracy gain proportional?
- The MLP sees a flattened 784-vector, so pixel *position* is lost. Which organ classes
  does the confusion matrix show it struggling with, and does AlexNet fix those?
- **To do (slides):** implement VGG16, ResNet18, ResNet50, DenseNet, EfficientNet and
  compare. `torchvision.models` gives you all of these in one line.